In [10]:
import pandas as pd
from collections import defaultdict
from itertools import chain, combinations

# Node class for FP-tree
class FPNode:
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None

    def increment(self, count):
        self.count += count

class FPTree:
    def __init__(self, transactions, min_support):
        self.root = FPNode(None, 1, None)
        self.headers = defaultdict(list)
        self.min_support = min_support
        self.frequent_items = self.find_frequent_items(transactions)
        self.build_tree(transactions)

    def find_frequent_items(self, transactions):
        item_counts = defaultdict(int)
        for transaction in transactions:
            for item in transaction:
                item_counts[item] += 1
        return {item: count for item, count in item_counts.items() if count >= self.min_support}

    def build_tree(self, transactions):
        for transaction in transactions:
            sorted_items = [item for item in sorted(transaction, key=lambda item: self.frequent_items.get(item, 0), reverse=True) if item in self.frequent_items]
            self.insert_tree(sorted_items, self.root)

    def insert_tree(self, items, node):
        if items:
            first_item = items[0]
            if first_item in node.children:
                node.children[first_item].increment(1)
            else:
                new_node = FPNode(first_item, 1, node)
                node.children[first_item] = new_node
                self.update_headers(first_item, new_node)

            remaining_items = items[1:]
            self.insert_tree(remaining_items, node.children[first_item])

    def update_headers(self, item, node):
        if self.headers[item]:
            current_node = self.headers[item][0]
            while current_node.link:
                current_node = current_node.link
            current_node.link = node
        else:
            self.headers[item].append(node)

    def mine_patterns(self, suffix):
        patterns = {}
        items = [item for item in self.headers.keys() if self.frequent_items[item] >= self.min_support]
        for item in items:
            new_suffix = suffix.copy()
            new_suffix.add(item)
            patterns[frozenset(new_suffix)] = self.frequent_items[item]

            conditional_tree_input = []
            node = self.headers[item][0]
            while node:
                path = []
                parent = node.parent
                while parent.parent:
                    path.append(parent.item)
                    parent = parent.parent
                conditional_tree_input.extend([path] * node.count)
                node = node.link

            conditional_tree = FPTree(conditional_tree_input, self.min_support)
            conditional_patterns = conditional_tree.mine_patterns(new_suffix)
            for pattern, count in conditional_patterns.items():
                patterns[pattern] = count
        return patterns

def find_frequent_itemsets(dataset: pd.DataFrame, min_support_count: int) -> list[set[str]]:
    transactions = dataset.apply(lambda row: row.index[row == 1].tolist(), axis=1).tolist()
    fp_tree = FPTree(transactions, min_support_count)
    patterns = fp_tree.mine_patterns(set())
    return [set(pattern) for pattern in patterns]

def generate_rules(frequent_itemsets: list[set[str]], min_confidence: float, dataset: pd.DataFrame) -> list[tuple[set[str], set[str]]]:
    def calculate_support(itemset):
        return dataset[list(itemset)].all(axis=1).sum()

    rules = []  
    for itemset in frequent_itemsets:
        for antecedent in chain(*[combinations(itemset, r) for r in range(1, len(itemset))]):
            antecedent = set(antecedent)
            consequent = itemset - antecedent
            if consequent:
                antecedent_support = calculate_support(antecedent)
                itemset_support = calculate_support(itemset)
                confidence = itemset_support / antecedent_support if antecedent_support > 0 else 0
                if confidence >= min_confidence:
                    rules.append((antecedent, consequent))
    return rules

def save_to_file(content, filename):
    with open(filename, 'w') as f:
        for item in content:
            f.write(f'{item}\n')

def main():
    # Load the preprocessed dataset
    df = pd.read_csv('../Dist/adult_preprocessed.csv')

    # Generate frequent itemsets
    frequent_itemsets = find_frequent_itemsets(df, min_support_count=13000)
    save_to_file([', '.join(itemset) for itemset in frequent_itemsets], '../Dist/freq_itemsets.txt')
    
    # Print number of frequent itemsets
    print(f'Number of frequent itemsets: {len(frequent_itemsets)}')

    # Generate association rules
    min_confidence = 0.95
    rules = generate_rules(frequent_itemsets, min_confidence, df)
    save_to_file([f"({', '.join(antecedent)}) -> ({', '.join(consequent)})" for antecedent, consequent in rules], '../Dist/rules.txt')

    # Print number of generated rules
    print(f'Number of generated rules: {len(rules)}')

if __name__ == '__main__':
    main()


Number of frequent itemsets: 164
Number of generated rules: 95
